# 02 — Domain Feature Engineering & Scikit-Learn Pipeline
**Project:** Insurance Claim Fraud Detection — Major Project Phase 1  
**Objective:** Build domain-specific indicators (Flash Claims, Claim-to-Premium ratios, Out-of-state incidents) and encapsulate everything inside a scikit-learn `Pipeline` and `ColumnTransformer` to eliminate data leakage.

In [ ]:
from pathlib import Path

import sys

import pandas as pd

import numpy as np



sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import load_raw_data, parse_dates_and_target, get_stratified_split

from src.features import InsuranceFeatureEngineer

from src.pipeline import build_preprocessor, build_fraud_pipeline

## 1. Domain Feature Engineering Demonstration
We extract high-signal indicators from dates, financial components, and geography:

In [ ]:
raw_df = load_raw_data()

clean_df = parse_dates_and_target(raw_df)

X_train, X_test, y_train, y_test = get_stratified_split(clean_df, test_size=0.20)



fe = InsuranceFeatureEngineer(drop_identifiers=True)

X_train_fe = fe.fit_transform(X_train)

print('New Engineered Features created:')

for col in ['days_to_incident', 'is_flash_claim', 'claim_to_premium_ratio', 'vehicle_claim_ratio', 'is_out_of_state_incident', 'is_night_incident']:

    print(f'  - {col}: sample value = {X_train_fe[col].iloc[0]}')

X_train_fe[['days_to_incident', 'is_flash_claim', 'claim_to_premium_ratio', 'is_out_of_state_incident']].head()

## 2. Inspecting the Preprocessor (Imputation & Scaling)
- Numerical: `SimpleImputer(median)` -> `RobustScaler()`
- Categorical: `SimpleImputer(constant='MISSING')` -> `OneHotEncoder(handle_unknown='ignore')`

In [ ]:
preprocessor = build_preprocessor()

X_preprocessed = preprocessor.fit_transform(X_train_fe)

print(f'Transformed Matrix Shape: {X_preprocessed.shape}')

print(f'Total Features after One-Hot Encoding: {X_preprocessed.shape[1]}')

## 3. End-to-End Pipeline Assembly
The complete pipeline accepts raw dataframes, performs feature extraction, encodes features, and feeds into the classifier.

In [ ]:
pipeline = build_fraud_pipeline()

pipeline.fit(X_train, y_train)

sample_preds = pipeline.predict(X_test[:5])

sample_probs = pipeline.predict_proba(X_test[:5])[:, 1]



print('Pipeline Sample Predictions:', sample_preds)

print('Pipeline Fraud Probabilities:', sample_probs.round(3))